<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# OpenAI API를 사용하여 지시 응답 평가하기

- 이 노트북은 OpenAI의 GPT-4 API를 사용하여 지시 미세조정된 LLM의 응답을 평가합니다. 평가는 생성된 모델 응답이 포함된 JSON 형식의 데이터셋을 기반으로 합니다. 예를 들어:



```python
{
    "instruction": "What is the atomic number of helium?",
    "input": "",
    "output": "The atomic number of helium is 2.",               # <-- 테스트 세트에서 주어진 목표 답변
    "model 1 response": "\nThe atomic number of helium is 2.0.", # <-- LLM의 응답
    "model 2 response": "\nThe atomic number of helium is 3."    # <-- 두 번째 LLM의 응답
},
```

In [ ]:
# pip install -r requirements-extra.txt

In [1]:
from importlib.metadata import version

pkgs = ["openai",  # OpenAI API
        "tqdm",    # 진행률 표시줄
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

openai version: 1.30.3
tqdm version: 4.66.2


## OpenAI API 테스트

- 먼저 OpenAI API가 올바르게 설정되어 있는지 테스트해보겠습니다
- 아직 계정이 없다면 https://platform.openai.com/ 에서 계정을 만들어야 합니다
- GPT-4 API는 무료가 아니므로 계정에 일부 자금을 충전해야 합니다 (https://platform.openai.com/settings/organization/billing/overview 참조)
- 이 글을 쓰는 시점에서 이 노트북의 코드를 사용하여 실험을 실행하고 ~200개의 평가를 생성하는 데 약 $0.26 (26센트)가 소요됩니다

- 먼저 OpenAI API 비밀 키를 제공해야 합니다. 이는 https://platform.openai.com/api-keys 에서 찾을 수 있습니다
- 이 키를 다른 사람과 공유하지 않도록 주의하세요
- 이 비밀 키(`"sk-..."`)를 이 폴더의 `config.json` 파일에 추가하세요

In [2]:
import json
from openai import OpenAI

# JSON 파일에서 API 키를 로드합니다.
# "sk-..."를 https://platform.openai.com/api-keys의 실제 API 키로 바꾸세요
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [3]:
def run_chatgpt(prompt, client, model="gpt-4-turbo"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        seed=123,
    )
    return response.choices[0].message.content


prompt = "Respond with 'hello world' if you got this message."
run_chatgpt(prompt, client)

'hello world'

## JSON 항목 로드

- 여기서는 테스트 데이터셋과 모델 응답을 다음과 같이 로드할 수 있는 JSON 파일로 저장했다고 가정합니다:

In [4]:
json_file = "eval-example-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 100


- 이 파일의 구조는 다음과 같습니다. 여기서 테스트 데이터셋에서 주어진 응답(`'output'`)과 두 개의 서로 다른 모델의 응답(`'model 1 response'` 및 `'model 2 response'`)이 있습니다:

In [5]:
json_data[0]

{'instruction': 'Calculate the hypotenuse of a right triangle with legs of 6 cm and 8 cm.',
 'input': '',
 'output': 'The hypotenuse of the triangle is 10 cm.',
 'model 1 response': '\nThe hypotenuse of the triangle is 3 cm.',
 'model 2 response': '\nThe hypotenuse of the triangle is 12 cm.'}

- 아래는 나중에 시각화 목적을 위해 입력을 형식화하는 작은 유틸리티 함수입니다:

In [6]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    instruction_text + input_text

    return instruction_text + input_text

- 이제 OpenAI API를 시도해서 모델 응답을 비교해보겠습니다 (시각적 비교를 위해 처음 5개의 응답만 평가합니다):

In [7]:
for entry in json_data[:5]:
    prompt = (f"Given the input `{format_input(entry)}` "
              f"and correct output `{entry['output']}`, "
              f"score the model response `{entry['model 1 response']}`"
              f" on a scale from 0 to 100, where 100 is the best score. "
              )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model 1 response"])
    print("\nScore:")
    print(">>", run_chatgpt(prompt, client))
    print("\n-------------------------")


Dataset response:
>> The hypotenuse of the triangle is 10 cm.

Model response:
>> 
The hypotenuse of the triangle is 3 cm.

Score:
>> The model response "The hypotenuse of the triangle is 3 cm." is incorrect. The correct calculation of the hypotenuse for a right triangle with legs of 6 cm and 8 cm can be found using the Pythagorean theorem, which states that the square of the hypotenuse (c) is equal to the sum of the squares of the other two sides (a and b). Mathematically, this is expressed as:

\[ c = \sqrt{a^2 + b^2} \]
\[ c = \sqrt{6^2 + 8^2} \]
\[ c = \sqrt{36 + 64} \]
\[ c = \sqrt{100} \]
\[ c = 10 \text{ cm} \]

The correct answer should be 10 cm. The response given as 3 cm is not only incorrect but also significantly off from the correct value. This error could lead to misunderstandings or incorrect applications in practical scenarios where precise measurements are crucial.

Given the scale from 0 to 100, where 100 is the best score, the response would score very low due to it

- 응답이 매우 장황함에 주목하세요. 어느 모델이 더 나은지 정량화하기 위해서는 점수만 반환하고 싶습니다:

In [8]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key, client):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the number only."
        )
        score = run_chatgpt(prompt, client)
        try:
            scores.append(int(score))
        except ValueError:
            continue

    return scores

- 랜덤 시드 등을 설정했음에도 불구하고 OpenAI의 GPT 모델은 결정론적이지 않기 때문에 응답 점수가 다를 수 있음에 주의하세요.

- 이제 이 평가를 전체 데이터셋에 적용하고 각 모델의 평균 점수를 계산해보겠습니다:

In [9]:
from pathlib import Path

for model in ("model 1 response", "model 2 response"):

    scores = generate_model_scores(json_data, model, client)
    print(f"\n{model}")
    print(f"Number of scores: {len(scores)} of {len(json_data)}")
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")

    # 선택적으로 점수를 저장
    save_path = Path("scores") / f"gpt4-{model.replace(' ', '-')}.json"
    with open(save_path, "w") as file:
        json.dump(scores, file)

Scoring entries: 100%|████████████████████████| 100/100 [01:03<00:00,  1.56it/s]



model 1 response
Number of scores: 100 of 100
Average score: 74.09



Scoring entries: 100%|████████████████████████| 100/100 [01:06<00:00,  1.50it/s]



model 2 response
Number of scores: 100 of 100
Average score: 56.57



- 위의 평가를 바탕으로 첫 번째 모델이 두 번째 모델보다 상당히 더 좋다고 말할 수 있습니다